# Bronze Layer - Ingestion Pipeline

Principles:
- Raw data only: no transformation, no filtering, no renaming
- Every run tagged with a unique RunID and ingestion timestamp
- Append-only: existing data is never modified
- Source data saved as-is in Parquet format (preserves types)

## Sources
1. OFS Unemployment BIT (CSV) → bronze_unemployment.parquet
2. SNB CPI (API JSON) → bronze_cpi.parquet  
3. SNB Policy Rate (API JSON) → bronze_policy_rate.parquet

In [1]:
import pandas as pd
import requests
import uuid
from datetime import datetime, timezone
from pathlib import Path

# Bronze output directory
BRONZE_PATH = Path("../data/bronze")
BRONZE_PATH.mkdir(parents=True, exist_ok=True)

# Run metadata - generated once per pipeline execution
RUN_ID = str(uuid.uuid4())
INGESTED_AT = datetime.now(timezone.utc).isoformat()

print(f"Run ID   : {RUN_ID}")
print(f"Timestamp: {INGESTED_AT}")

Run ID   : 88455a82-f70b-4392-b459-1c2e7a0f0dac
Timestamp: 2026-07-15T12:59:47.583578+00:00


In [2]:
# ── Source 1: OFS Unemployment ──────────────────────────────────────────────

def ingest_unemployment(run_id: str, ingested_at: str) -> pd.DataFrame:
    """
    Download OFS unemployment CSV and tag with run metadata.
    No transformation applied - raw data only.
    """
    url = "https://dam-api.bfs.admin.ch/hub/api/dam/assets/36519062/master"
    
    df = pd.read_csv(url, sep=",", encoding="utf-8")
    
    # Add run metadata columns
    df["_run_id"] = run_id
    df["_ingested_at"] = ingested_at
    df["_source"] = "ofs_unemployment_bit"
    df["_source_url"] = url
    
    return df


df_unemployment = ingest_unemployment(RUN_ID, INGESTED_AT)

print(f"Rows ingested : {len(df_unemployment)}")
print(f"Columns       : {df_unemployment.columns.tolist()}")
print(f"\nMetadata sample:")
print(df_unemployment[["_run_id", "_ingested_at", "_source"]].iloc[0])

Rows ingested : 9888
Columns       : ['INDICATORS_HRCHY', 'INDICATORS_FR', 'INDICATORS_DE', 'GENDER_FR', 'GENDER_DE', 'DETAILS_FR', 'DETAILS_DE', 'PERIOD', 'FREQ', 'MEASURE_FR', 'MEASURE_DE', 'VALUE', 'STATUS', 'STATUS_1', '_run_id', '_ingested_at', '_source', '_source_url']

Metadata sample:
_run_id         88455a82-f70b-4392-b459-1c2e7a0f0dac
_ingested_at        2026-07-15T12:59:47.583578+00:00
_source                         ofs_unemployment_bit
Name: 0, dtype: object


In [3]:
# ── Source 2: SNB CPI ────────────────────────────────────────────────────────

def ingest_cpi(run_id: str, ingested_at: str) -> pd.DataFrame:
    """
    Fetch SNB CPI API (JSON) and flatten both timeseries into a DataFrame.
    No transformation applied - raw data only.
    """
    url = "https://data.snb.ch/api/cube/plkopr/data/json/fr"
    
    response = requests.get(url)
    response.raise_for_status()
    
    dfs = []
    for ts in response.json()["timeseries"]:
        series_key = ts["metadata"]["key"]
        series_name = ts["header"][0]["dimItem"]
        df_ts = pd.DataFrame(ts["values"])
        df_ts["_series_key"] = series_key
        df_ts["_series_name"] = series_name
        dfs.append(df_ts)
    
    df = pd.concat(dfs, ignore_index=True)
    
    # Add run metadata columns
    df["_run_id"] = run_id
    df["_ingested_at"] = ingested_at
    df["_source"] = "snb_cpi"
    df["_source_url"] = url
    
    return df


df_cpi = ingest_cpi(RUN_ID, INGESTED_AT)

print(f"Rows ingested : {len(df_cpi)}")
print(f"Columns       : {df_cpi.columns.tolist()}")
print(f"\nSample:\n{df_cpi.head(3)}")

Rows ingested : 2518
Columns       : ['date', 'value', '_series_key', '_series_name', '_run_id', '_ingested_at', '_source', '_source_url']

Sample:
      date      value                _series_key  \
0  1921-01  19.588398  EPB@SNB.plkopr{LD2010100}   
1  1921-02  19.361351  EPB@SNB.plkopr{LD2010100}   
2  1921-03  19.098022  EPB@SNB.plkopr{LD2010100}   

                          _series_name                               _run_id  \
0  Indice suisse – Décembre 2025 = 100  88455a82-f70b-4392-b459-1c2e7a0f0dac   
1  Indice suisse – Décembre 2025 = 100  88455a82-f70b-4392-b459-1c2e7a0f0dac   
2  Indice suisse – Décembre 2025 = 100  88455a82-f70b-4392-b459-1c2e7a0f0dac   

                       _ingested_at  _source  \
0  2026-07-15T12:59:47.583578+00:00  snb_cpi   
1  2026-07-15T12:59:47.583578+00:00  snb_cpi   
2  2026-07-15T12:59:47.583578+00:00  snb_cpi   

                                        _source_url  
0  https://data.snb.ch/api/cube/plkopr/data/json/fr  
1  https://data.snb.c

In [4]:
# ── Source 3: SNB Policy Rate ────────────────────────────────────────────────

def ingest_policy_rate(run_id: str, ingested_at: str) -> pd.DataFrame:
    """
    Fetch SNB policy rate API (JSON) and flatten all timeseries.
    All 10 series ingested raw - filtering to Swiss rate only happens in Silver.
    """
    url = "https://data.snb.ch/api/cube/snboffzisa/data/json/fr"
    
    response = requests.get(url)
    response.raise_for_status()
    
    dfs = []
    for ts in response.json()["timeseries"]:
        series_key = ts["metadata"]["key"]
        series_name = ts["header"][0]["dimItem"]
        df_ts = pd.DataFrame(ts["values"])
        df_ts["_series_key"] = series_key
        df_ts["_series_name"] = series_name
        dfs.append(df_ts)
    
    df = pd.concat(dfs, ignore_index=True)
    
    # Add run metadata columns
    df["_run_id"] = run_id
    df["_ingested_at"] = ingested_at
    df["_source"] = "snb_policy_rate"
    df["_source_url"] = url
    
    return df


df_policy = ingest_policy_rate(RUN_ID, INGESTED_AT)

print(f"Rows ingested : {len(df_policy)}")
print(f"\nSeries ingested:")
print(df_policy.groupby("_series_name")["date"].agg(["min", "max", "count"]))

Rows ingested : 2710

Series ingested:
                                                        min      max  count
_series_name                                                               
Japon - Banque du Japon - Taux directeur            2000-01  2026-05    258
Royaume-Uni - Banque d’Angleterre - Taux directeur  2000-01  2026-05    317
Suisse - BNS - Marge de fluctuation du Libor à ...  2000-01  2019-05    233
Suisse - BNS - Marge de fluctuation du Libor à ...  2000-01  2019-05    233
Suisse - Taux directeur de la BNS                   2019-06  2026-05     84
Zone euro/BCE - Facilité de dépôt                   2000-01  2026-05    317
Zone euro/BCE - Facilité de prêt marginal           2000-01  2026-05    317
Zone euro/BCE - Opérations principales de refin...  2000-01  2026-05    317
États-Unis - Fed - Marge de fluctuation - limit...  2000-01  2026-05    317
États-Unis - Fed - Marge de fluctuation - limit...  2000-01  2026-05    317


In [5]:
# ── Save to Bronze (Parquet) ─────────────────────────────────────────────────

def save_bronze(df: pd.DataFrame, name: str, path: Path) -> None:
    """Save DataFrame to Parquet in bronze directory."""
    output_path = path / f"bronze_{name}.parquet"
    df.to_parquet(output_path, index=False)
    print(f"Saved {len(df)} rows → {output_path}")


save_bronze(df_unemployment, "unemployment", BRONZE_PATH)
save_bronze(df_cpi, "cpi", BRONZE_PATH)
save_bronze(df_policy, "policy_rate", BRONZE_PATH)

print(f"\nBronze layer complete - Run ID: {RUN_ID}")

Saved 9888 rows → ..\data\bronze\bronze_unemployment.parquet
Saved 2518 rows → ..\data\bronze\bronze_cpi.parquet
Saved 2710 rows → ..\data\bronze\bronze_policy_rate.parquet

Bronze layer complete - Run ID: 88455a82-f70b-4392-b459-1c2e7a0f0dac


## Bronze Ingestion - Summary

| Source | Rows | File |
|--------|------|------|
| OFS Unemployment BIT | 9,888 | bronze_unemployment.parquet |
| SNB CPI | 2,518 | bronze_cpi.parquet |
| SNB Policy Rate | 2,710 | bronze_policy_rate.parquet |

Run ID: 88455a82-f70b-4392-b459-1c2e7a0f0dac  
All sources ingested raw - no transformation applied.  
Next step: Silver layer - filtering, typing, reconstruction.